# 02 — Classical ML Models

**Models:** Random Forest · SVM · XGBoost · Neural Network  
**Task:** 5-class climate condition classification  
**Evaluation:** Accuracy, F1 macro, confusion matrices, scalability

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import yaml

from src.features.engineering import (
    load_config, prepare_splits, prepare_quantum_subset, save_artifacts
)
from src.models.classical import ClassicalModelTrainer
from src.evaluation.metrics import evaluate_model, compare_models, print_comparison_table
from src.data.label import LABEL_MAP

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'figure.dpi': 150,
})

config = load_config('../config/config.yaml')
CLASS_NAMES = config['labeling']['class_names']
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Classes: {CLASS_NAMES}")

## 1. Load Data & Prepare Splits

In [ ]:
df = pd.read_csv('../data/processed/merra2_india_labeled.csv')
print(f"Loaded {len(df)} samples, {df['label'].nunique()} classes")

splits = prepare_splits(df, config)
X_train = splits['X_train']
X_val   = splits['X_val']
X_test  = splits['X_test']
y_train = splits['y_train']
y_val   = splits['y_val']
y_test  = splits['y_test']
feature_names = splits['feature_names']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Features ({len(feature_names)}): {feature_names}")

In [ ]:
# Prepare and save quantum subset (needed for Phase 3)
quantum = prepare_quantum_subset(
    X_train, y_train, X_test, y_test,
    n_components=config['quantum_ml']['pca_components'],
    subset_size=config['quantum_ml']['training_subset_size'],
    random_state=config['data_split']['random_state'],
)
save_artifacts(splits, quantum, config)
print(f"PCA explained variance: {quantum['pca_explained_variance']:.1%}")
print(f"Quantum train: {quantum['X_train_q'].shape}")

## 2. Train All Classical Models

In [ ]:
trainer = ClassicalModelTrainer(config)
train_results = trainer.train_all(X_train, y_train, X_val, y_val)

print("\nTraining complete:")
for name, r in train_results.items():
    print(f"  {name:20s}: {r['training_time']:.1f}s")

## 3. Evaluate on Test Set

In [ ]:
from pathlib import Path

test_results = {}
model_display_names = {
    'random_forest': 'Random Forest',
    'svm': 'SVM (RBF)',
    'xgboost': 'XGBoost',
    'neural_network': 'Neural Network',
}

for model_key, display_name in model_display_names.items():
    metrics = evaluate_model(
        predict_fn=lambda X, k=model_key: trainer.predict(k, X),
        X_test=X_test,
        y_test=y_test,
        model_name=display_name,
        class_names=CLASS_NAMES,
    )
    metrics['training_time'] = train_results[model_key]['training_time']
    test_results[display_name] = metrics
    print(f"\n{display_name}:")
    print(metrics['classification_report'])

trainer.save_models(Path('../results/models'))

In [ ]:
comparison_df = compare_models(test_results)
print_comparison_table(comparison_df)
comparison_df.to_csv('../results/classical_comparison.csv', index=False)

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (name, r) in zip(axes, test_results.items()):
    cm = r['confusion_matrix'].astype(float)
    # Normalize by true class (row)
    cm_norm = cm / (cm.sum(axis=1, keepdims=True) + 1e-10)
    n_classes = cm_norm.shape[0]
    labels = CLASS_NAMES[:n_classes]

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(n_classes))
    ax.set_yticks(range(n_classes))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{name}\nAcc={r["accuracy"]:.3f} F1={r["f1_macro"]:.3f}')

    for i in range(n_classes):
        for j in range(n_classes):
            val = cm_norm[i, j]
            ax.text(j, i, f'{val:.2f}',
                    ha='center', va='center',
                    color='white' if val > 0.5 else 'black',
                    fontsize=8)

    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Confusion Matrices — Classical Models (row-normalized)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/classical_confusion_matrices.png',
            dpi=200, bbox_inches='tight')
plt.show()

## 5. Performance Comparison

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Accuracy & F1 Score', 'Training Time (seconds)'])

models = comparison_df['model'].tolist()
x = list(range(len(models)))

fig.add_trace(go.Bar(name='Accuracy', x=models, y=comparison_df['accuracy'],
                      marker_color='#2196F3', text=comparison_df['accuracy'].round(3),
                      textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(name='F1 Macro', x=models, y=comparison_df['f1_macro'],
                      marker_color='#4CAF50', text=comparison_df['f1_macro'].round(3),
                      textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(name='Training Time', x=comparison_df['model'],
                      y=comparison_df['training_time_s'],
                      marker_color='#FF9800',
                      text=comparison_df['training_time_s'].round(1),
                      textposition='outside'), row=1, col=2)

fig.update_layout(title='Classical Model Comparison', height=450,
                   barmode='group', showlegend=True)
fig.update_yaxes(range=[0, 1.1], row=1, col=1)
fig.show()

## 6. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_key, title in [
    (axes[0], 'random_forest', 'Random Forest Feature Importance'),
    (axes[1], 'xgboost', 'XGBoost Feature Importance'),
]:
    if 'feature_importances' not in train_results[model_key]:
        ax.set_visible(False)
        continue
    importances = train_results[model_key]['feature_importances']
    # Top 12 features
    top_idx = np.argsort(importances)[-12:]
    top_vals = importances[top_idx]
    top_names = [feature_names[i] for i in top_idx]

    ax.barh(top_names, top_vals, color='#2196F3', edgecolor='black', linewidth=0.4)
    ax.set_xlabel('Importance')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('../results/figures/feature_importance.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Neural Network Learning Curves

In [ ]:
nn_result = train_results['neural_network']
epochs = range(1, len(nn_result['train_losses']) + 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(epochs), y=nn_result['train_losses'],
                          name='Train Loss', line=dict(color='#2196F3', width=2)))
fig.add_trace(go.Scatter(x=list(epochs), y=nn_result['val_losses'],
                          name='Val Loss', line=dict(color='#F44336', width=2,
                                                       dash='dash')))
fig.update_layout(
    title='Neural Network Training Curves',
    xaxis_title='Epoch', yaxis_title='Cross-Entropy Loss',
    height=400, legend=dict(x=0.7, y=0.9)
)
fig.show()

## 8. Scalability Analysis

In [ ]:
scaling_df = trainer.scalability_analysis(X_train, y_train, X_test, y_test)
scaling_df.to_csv('../results/classical_scaling.csv', index=False)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['F1 Score vs Training Size',
                                    'Training Time vs Dataset Size'])

colors = {'Random Forest': '#2196F3', 'XGBoost': '#FF9800'}
for model_name in scaling_df['model'].unique():
    sub = scaling_df[scaling_df['model'] == model_name]
    fig.add_trace(
        go.Scatter(x=sub['train_size'], y=sub['f1_macro'],
                   name=model_name, mode='lines+markers',
                   line=dict(color=colors.get(model_name, 'gray'), width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=sub['train_size'], y=sub['training_time'],
                   name=model_name, mode='lines+markers',
                   line=dict(color=colors.get(model_name, 'gray'), width=2),
                   showlegend=False),
        row=1, col=2
    )

fig.update_layout(title='Classical Model Scalability', height=400)
fig.update_xaxes(title_text='Training samples', row=1, col=1)
fig.update_xaxes(title_text='Training samples', row=1, col=2)
fig.update_yaxes(title_text='F1 Macro', row=1, col=1)
fig.update_yaxes(title_text='Time (s)', row=1, col=2)
fig.show()

## Summary

**Next:** Phase 3 — Quantum ML (QSVC + VQC) on PCA-reduced 4-dimensional features.